## 🏦 Business Problem Definition

This project aims to investigate credit card transactions using descriptive and inferential statistical techniques. The analysis focuses on understanding transaction characteristics, identifying patterns associated with fraudulent activities, and applying probability distributions to real-world financial data.

The statistical analysis will be conducted using Python, with particular attention to the Binomial, Poisson, and Normal distributions. The findings will be interpreted to evaluate the suitability of each statistical distribution for describing the selected dataset.

**🎯 Objectives:**

- Explore and understand the structure of the dataset.
- Perform descriptive statistical analysis.
- Investigate the distribution of fraudulent and
  legitimate transactions.
- Apply the Binomial distribution to model
  fraudulent transaction occurrences.
- Apply the Poisson distribution to investigate
  transaction counts over defined time intervals.
- Investigate the suitability of the Normal
  distribution for selected numerical variables.
- Compare and critically evaluate the statistical
  distributions used in the analysis.

### 🔄 Download the libraries if you don't have them!

In [1]:
# !pip install -r ../requirements.txt

### 🤔 JupyterLab Authoring ©

In [2]:
%reload_ext watermark
%watermark --author "Matheus DoS S. Silva"
%watermark --email "2026438@student.cct.ie"
%watermark --github_username "MattheuSxS"
%watermark --date
%watermark --python

Author: Matheus DoS S. Silva

Email: 2026438@student.cct.ie

Github username: MattheuSxS

Date: 2026-10-31

Python implementation: CPython
Python version       : 3.14.9
IPython version      : 9.17.1



### 🛂 Importing Packages

This is the workspace preparation phase. Here, we load all the necessary Python tools and libraries (such as Pandas for data manipulation, Scikit-learn for modeling, and Matplotlib/Seaborn for visualizations) that will be used throughout the project.

In [3]:
import logging
import kagglehub
import numpy as np
import pandas as pd        # Data manipulation and analysis.
import seaborn as sns
from scipy import stats
from pathlib import Path
import matplotlib.pyplot as plt


# Display settings
sns.set_style('whitegrid')
%matplotlib inline

In [4]:
logging.basicConfig(
    format=("%(asctime)s | %(levelname)s | File_name ~> %(module)s.py "
            "| Function ~> %(funcName)s | Line ~~> %(lineno)d  ~~>  %(message)s"),
    level=logging.INFO
)

### 🔄 Data Loading and Initial Inspection

This is the first encounter with the project's raw material. We load the data from a source (in our case, a CSV file) into a Pandas DataFrame and perform an initial check to understand its structure.

The download_dataset() function is responsible for loading the dataset used in the Machine Learning project. 
* **1 ~~>** It first checks whether the dataset already exists locally in the ../data directory.
* **2 ~~>** If the file is not found, it attempts to download the dataset from the kaggle using the kagglehub function.
 
Once the dataset is available, the function loads it into a Pandas DataFrame and returns it for further data analysis and Machine Learning processing. The function also includes logging and exception handling to provide information about the download and loading process.

In [ ]:
def download_dataset() -> pd.DataFrame:
    data_dir = Path("../data")
    dataset_file = data_dir / "creditcard.csv"

    data_dir.mkdir(parents=True, exist_ok=True)

    try:
        if not dataset_file.exists():

            logging.warning("📥 Dataset not found. Downloading from Kaggle...")

            path = kagglehub.dataset_download(
                "mlg-ulb/creditcardfraud",
                output_dir=str(data_dir)
            )

            logging.info(f"✅ Dataset downloaded to: {path}")

        else:

            logging.info(f"✅ Dataset already exists: {dataset_file}")

        return  pd.read_csv(dataset_file, sep=",")

    except Exception as e:
        logging.error(f"❌ Failed to download or read the dataset: {e}")

In [6]:
df = download_dataset()

2026-10-31 17:36:45,591 | WARNING | File_name ~> 4256853042.py | Function ~> download_dataset | Line ~~> 10  ~~>  📥 Dataset not found. Downloading from Kaggle...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 66.0M/66.0M [00:19<00:00, 3.61MB/s]

Extracting files...



2026-10-31 17:37:06,181 | INFO | File_name ~> 4256853042.py | Function ~> download_dataset | Line ~~> 17  ~~>  ✅ Dataset downloaded to: ../data


In [7]:
print(f"Rows: {df.shape[0]:,} ~ Columns: {df.shape[1]}")

Rows: 284,807 ~ Columns: 31


**📍 Variable Information:**

* **Time:** Seconds elapsed between each transaction and the first transaction.
* **V1–V28:** Anonymised numerical features obtained through Principal Component Analysis (PCA).
* **Amount:** Transaction amount.
* **Class:** Target variable identifying fraudulent (1) and legitimate (0) transactions.

In [9]:
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [10]:
df.sample(3)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
153920,100654.0,1.959818,0.447708,-2.392257,1.494472,1.282000,-0.336082,0.478084,-0.221582,1.244884,...,-0.152951,-0.113375,-0.069246,0.025431,0.460284,-0.552584,-0.041234,-0.038704,39.46,0
15154,26501.0,-0.413410,0.990354,1.548977,-0.024993,-0.134042,-0.917827,0.646647,-0.021225,-0.276001,...,-0.237651,-0.659646,-0.015045,0.339432,-0.220263,0.074161,0.260447,0.119249,1.78,0
27110,34393.0,1.250720,-0.883256,1.320847,-0.514008,-1.846829,-0.470431,-1.197293,0.009880,-0.266866,...,-0.135041,0.047471,0.036999,0.447709,-0.045371,1.108492,-0.007613,0.035423,49.95,0


In [11]:
df.tail(5)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0
284806,172792.0,-0.533413,-0.189733,0.703337,-0.506271,-0.012546,-0.649617,1.577006,-0.414650,0.486180,...,0.261057,0.643078,0.376777,0.008797,-0.473649,-0.818267,-0.002415,0.013649,217.00,0


In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     284807 non-nu

In [13]:
df.isnull().sum()

Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64

**⚠⚠ Important: ⚠⚠**

I would not automatically remove repeated rows. The same Creditcard Fraud may appear across different time windows, and this does not necessarily mean that the records are improper duplicates.

In [14]:
df.duplicated().sum()

np.int64(1081)

In [15]:
df.nunique()

Time      124592
V1        275663
V2        275663
V3        275663
V4        275663
V5        275663
V6        275663
V7        275663
V8        275663
V9        275663
V10       275663
V11       275663
V12       275663
V13       275663
V14       275663
V15       275663
V16       275663
V17       275663
V18       275663
V19       275663
V20       275663
V21       275663
V22       275663
V23       275663
V24       275663
V25       275663
V26       275663
V27       275663
V28       275663
Amount     32767
Class          2
dtype: int64

### 🕵🏾 Exploratory Data Analysis (EDA)

* **Transaction Amount Analysis:** Examine transaction amounts and compare their statistical characteristics across both classes.
* **Temporal Analysis:** Investigate transaction activity using the Time variable and identify potential patterns over time.
* **Data Visualization:** Use appropriate visualizations to communicate the main characteristics of the dataset.

### 👨🏾‍🔬 Probability Distributions

#### 🔬 Binomial Distribution

The Binomial distribution will be investigated using the binary Class variable.

Each transaction represents a trial with two possible outcomes:

- Success: Fraudulent transaction (Class = 1).
- Failure: Legitimate transaction (Class = 0).

The probability of fraud will be estimated from the observed proportion of fraudulent transactions.

The analysis will investigate:

- The probability of fraudulent transactions.
- Expected number of fraudulent transactions
  in a fixed sample.
- Variance and standard deviation.
- Probability of observing a specific number
  of fraudulent transactions.
- The behaviour of the distribution as sample
  size increases.

The theoretical results will be compared with observed data.

#### 🔬 Poisson Distribution

The Poisson distribution will be investigated to model the number of transactions occurring within fixed time intervals. The Time variable will be used to organise transactions into defined intervals.

The analysis will investigate:

- Transaction counts per time interval.
- Average event rate (lambda).
- Probability of observing a specific number
  of transactions.
- Comparison between observed and theoretical
  frequencies.
- The behaviour of the distribution with
  larger samples.

The assumptions of the Poisson distribution, including independence and a reasonably stable event rate, will be critically evaluated. The suitability of the model will be discussed based on the observed data.

#### 🔬 Normal Distribution

This section investigates the suitability of the Normal distribution for selected continuous numerical variables.

The analysis will focus on:

- Mean and standard deviation.
- Distribution shape.
- Skewness and kurtosis.
- Histogram and density visualization.
- Comparison between empirical and theoretical
  Normal distributions.
- Probability calculations using the Normal model.

The selected variables will be evaluated to determine whether the Normal distribution provides a reasonable approximation. Any deviations from Normality will be discussed and interpreted.

### ⚖ Statistical Comparison

This section compares the discrete and continuous probability distributions applied throughout the analysis.

The discussion will cover:

- Differences between discrete and continuous
  probability distributions.
- Suitability of the Binomial distribution
  for binary transaction outcomes.
- Suitability of the Poisson distribution
  for transaction counts.
- Suitability of the Normal distribution
  for continuous numerical variables.
- Impact of skewness and class imbalance.
- Limitations and assumptions of each model.

🎯 The objective is to justify the selection of each distribution and explain its relevance to the dataset.

### 📊 Conclusion